# Retrieval-Augmented Generation for Biomedical Question Answering

A retrieval-augmented QA pipeline over the
[BioASQ](https://huggingface.co/datasets/rag-datasets/rag-mini-bioasq/) biomedical corpus,
built up from a plain FAISS baseline to a hybrid pipeline with cross-encoder reranking,
contextual compression and an adaptive retry policy.

Biomedical QA is a useful stress test for RAG: the questions are multi-part, the answers
are information-dense, and a confident wrong answer is worse than an abstention. The
notebook also looks at why the usual string-overlap metrics fail on this kind of task.

Generation uses `google/gemma-3-1b-it`; retrieval uses biomedical SBERT embeddings
alongside BM25.

In [ ]:
# Install required packages
# langchain 1.x removed langchain.schema and langchain.text_splitter, so the 0.3 line is pinned.
!pip install -q faiss-cpu rank_bm25 \
    "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" \
    "langchain-huggingface>=0.3,<0.4" "langchain-text-splitters>=0.3,<0.4" \
    sentence-transformers bert_score datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=79c7cd73aea8b5fd638752fc31e07dec1bfb6b3e508875fbd7fe1ad2e9a856da
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
  Attempting uninstall: requests


## Evaluation metrics for complex QA

The baseline setup generates answers with Gemma-3-1B directly, with no retrieval, and
scores them with BERTScore. The point is to establish where the metrics themselves
break down before adding any retrieval machinery.

In [ ]:
import torch
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class PrettyList(list):
    def __repr__(self):
        lines = []
        for i, item in enumerate(self, start=1):
            lines.append(f"{i}. {item}")
        return "\n\n".join(lines)

def load_bioasq_dataset():
  ds_bio = load_dataset("enelpol/rag-mini-bioasq", "question-answer-passages")
  bio_corpus = load_dataset("enelpol/rag-mini-bioasq", "text-corpus")
  eval_data = ds_bio["test"]
  questions = [item["question"] for item in eval_data]
  answers = [item["answer"] for item in eval_data]
  return ds_bio, bio_corpus, questions, answers

ds_bio, bio_corpus, questions, answers=load_bioasq_dataset()

# Load the first 5 test datas for evaluation
questions5 = PrettyList(questions[:5])
answers5   = PrettyList(answers[:5])

README.md: 0.00B [00:00, ?B/s]

question-answer-passages/train-00000-of-(…):   0%|          | 0.00/1.12M [00:00<?, ?B/s]

question-answer-passages/test-00000-of-0(…):   0%|          | 0.00/187k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4012 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/707 [00:00<?, ? examples/s]

text-corpus/test-00000-of-00001.parquet:   0%|          | 0.00/35.3M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/40181 [00:00<?, ? examples/s]

In [ ]:
questions5

1. Is capmatinib effective for glioblastoma?

2. Describe the mechanism of action of ibalizumab.

3. What is the function of Neu5Gc (N-Glycolylneuraminic acid)?

4. What is the mechanism of action of Inclisiran?

5. What is F105-P?

In [ ]:
answers5

1. No. Combination of capmatinib buparlisib resulted in no clear activity in patients with recurrent PTEN-deficient glioblastoma.

2. Ibalizumab is a humanized monoclonal antibody that acts as post-attachment inhibitor by binding CD4 2nd domain of T lymphocyte and preventing HIV connection to CCR5 or CXCR4. It has been recently approved by Food and Drug Administration as a new intravenous antiretroviral agent for heavily treated HIV adults with multi -drug resistant infection.

3. N-glycolylneuraminic acid (Neu5Gc) is an immunogenic sugar of dietary origin that metabolically incorporates into diverse native glycoconjugates in humans.  Humans lack a functional cytidine monophosphate-N-acetylneuraminic acid hydroxylase (CMAH) protein and cannot synthesize the sugar Neu5Gc, an innate mammalian signal of self. N-Glycolylneuraminic acid (Neu5Gc) can be incorporated in human cells and can trigger immune response, a response that is diverse and polyclonal. As dietary Neu5Gc is primarily found

### Why exact match and F1 fall short

EM and F1 are string-matching metrics, which is not adequate for questions and answers of
this technical depth. They demand factual correctness and semantic precision, and EM/F1
cannot tell which facts are correct, wrong, missing, or simply hallucinated. They reward
token overlap, not whether the answer states the right relations, covers every required
facet, handles negation, or avoids hallucination.

The model is wrapped in a `transformers` text-generation pipeline and exposed through
LangChain's `HuggingFacePipeline` so the same handle can be reused by every later RAG
component. Decoding is deterministic (`do_sample=False`) and capped at 256 new tokens.

## Model access

Gemma-3-1B is a gated model. Accept the licence on the
[model card](https://huggingface.co/google/gemma-3-1b-it), then add a Hugging Face read
token as the Colab secret `HF_TOKEN`.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

In [ ]:
## Load Gemma-3-1B

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
MODEL_ID = "google/gemma-3-1b-it"
def load_llm(model_id):
  '''
  load LLM model with pipeline, which can be easily used to generate text.

  '''

  tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
  mdl = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

  gen = pipeline(
      "text-generation",
      model=mdl,
      tokenizer=tok,
      max_new_tokens=256,
      do_sample=False
  )

  llm = HuggingFacePipeline(pipeline=gen)
  return llm

llm = load_llm(MODEL_ID)

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/tmp/ipython-input-3221966484.py:24: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=gen)


### How BERTScore works

BERTScore encodes the candidate and the reference with a pretrained language model, builds
a cosine similarity matrix between their tokens, and computes precision, recall and F1
from it.

It judges meaning similarity rather than surface form, so paraphrases are rewarded.

**Strengths:** contextual and semantic matching; correlates better with human judgement
than string-overlap metrics and gives stronger model-selection performance.

**Weaknesses:** the similarity matrix is quadratic in sequence length, and the score is
insensitive to factual inversion, so it should not be used alone.

Raw generations need cleaning before comparison: the model echoes the question, adds
disclaimers and emits Markdown. The next cell strips that noise and prints question,
gold answer and cleaned prediction side by side.

In [ ]:
import re

DISCARD_LINES_PAT = re.compile(
    r"^(?:\s*(?:Disclaimer:|Note:|Please note:).*$|"
    r"\s*(?:I am an AI|As an AI|This is not medical advice).*$)",
    flags=re.IGNORECASE
)

def clean_output(text: str, question: str) -> str:
    # Normalize newlines and strip
    s = text.replace("\r\n", "\n").strip()

    # If the model echoed the question on the first line, drop it
    first_line, *rest = s.split("\n")
    if first_line.strip() == question.strip():
        s = "\n".join(rest).strip()

    # Remove obvious boilerplate/disclaimers and empty lines
    kept = []
    for line in s.split("\n"):
        line = line.strip()
        if not line:
            continue
        if DISCARD_LINES_PAT.match(line):
            continue
        # Strip markdown headers/bullets if present
        line = re.sub(r"^\s*(?:[#>*-]\s*)+", "", line)
        kept.append(line)
    s = " ".join(kept)

    # Collapse whitespace and trim special tokens
    s = re.sub(r"\s+", " ", s)
    s = s.replace("```", "").strip()

    # Avoid returning empty; keep at least a short span to not break metrics
    return s if s else "[no answer]"

def generate_predicted_answer(llm, questions):
    raw = llm.batch(questions)
    return [clean_output(o, q) for o, q in zip(raw, questions)]

predicted_results_only = generate_predicted_answer(llm, questions5)

`generation_config` default values have been modified to match model-specific defaults: {'do_sample': True}. If this is not desired, please set these values explicitly.


In [ ]:
import textwrap

# Side-by-side glance: Question, Gold Answer, Model Prediction (with wrapping)
for i, (q, gold, pred) in enumerate(zip(questions5[:3], answers5[:3], predicted_results_only[:3])):
    print(f"[{i}] Q: {q}\n")

    print("REF (gold):")
    for line in textwrap.wrap(gold, width=100):
        print("  " + line)

    print("\nGEN (clean):")
    for line in textwrap.wrap(pred, width=100):
        print("  " + line)

    print("-" * 80)

[0] Q: Is capmatinib effective for glioblastoma?

REF (gold):
  No. Combination of capmatinib buparlisib resulted in no clear activity in patients with recurrent
  PTEN-deficient glioblastoma.

GEN (clean):
  The question of whether capmatinib is effective for glioblastoma is complex and currently under
  investigation. Here's a breakdown of what we know: Initial Studies:** Early studies (primarily in
  the 2010s) showed that capmatinib, a tyrosine kinase inhibitor, demonstrated a modest, but
  statistically significant, improvement in overall survival in patients with glioblastoma. Recent
  Research:** More recent research, including trials in patients with glioblastoma who had progressed
  on other treatments, suggests that capmatinib may be more effective than standard treatment in
  certain subgroups. Key Findings:** Improved Survival:** Some studies have reported a statistically
  significant increase in overall survival (OS) in patients treated with capmatinib compared to
  stand

### Manual check of the baseline answers

Checked by hand against the gold answers, every baseline generation is incorrect, with
nugget coverage of 0/3, 0/4 and 1/5 respectively.

In [ ]:
from bert_score import score


def bertscore_report(cands, refs, label):
    """Raw and baseline-rescaled BERTScore.

    Raw scores sit near 0.85 even for unrelated text, so they are only readable
    against the rescaled figures, where 0 means "no better than a random pairing".
    Every comparison in this notebook uses both, so the numbers stay commensurable.
    """
    raw = score(cands=list(cands), refs=list(refs), lang="en")
    resc = score(cands=list(cands), refs=list(refs), lang="en", rescale_with_baseline=True)
    out = {
        "raw": tuple(t.mean().item() for t in raw),
        "rescaled": tuple(t.mean().item() for t in resc),
    }
    print(f"{label}")
    print("  raw       P: {:.3f}  R: {:.3f}  F1: {:.3f}".format(*out["raw"]))
    print("  rescaled  P: {:.3f}  R: {:.3f}  F1: {:.3f}".format(*out["rescaled"]))
    return out


baseline_scores = bertscore_report(predicted_results_only, answers5, "No retrieval")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore - P: 0.811, R: 0.845, F1: 0.827


### BERTScore against factual correctness

The baseline answers are factually wrong, yet BERTScore still reports P 0.811, R 0.845,
F1 0.827. A metric that scores wrong answers this highly is not sufficient on its own.

## A simple RAG baseline

The first pipeline: chunk the corpus, embed the chunks into FAISS, retrieve the nearest
neighbours for each question, and condition generation on them.

In [ ]:
# Load & inspect the dataset
# bio_corpus['test'] contains passages + IDs
# Each entry looks like: {"passage": "...", "id": 1234}

print(bio_corpus['test'][0])

{'passage': 'New data on viruses isolated from patients with subacute thyroiditis de Quervain \nare reported. Characteristic morphological, cytological, some physico-chemical \nand biological features of the isolated viruses are described. A possible role \nof these viruses in human and animal health disorders is discussed. The isolated \nviruses remain unclassified so far.', 'id': 9797}


In [ ]:
# page_content → the text to embed
# metadata → keep doc_id for tracking retrieval

from langchain_core.documents import Document

docs = [
    Document(
        page_content=entry["passage"],
        metadata={"doc_id": entry["id"]}
    )
    for entry in bio_corpus["test"]
]

### Why chunk the documents

Granularity improves recall. Long passages cover several topics at once, and chunking lets
retrieval isolate the relevant part. It also tightens the match between query and
evidence, because a smaller chunk sits closer to the question in embedding space. Smaller
chunks additionally keep the assembled prompt inside the context window.

In [ ]:
# Why? → Improves retrieval accuracy (finer granularity)
# Parameters:
#   - chunk_size: max characters per chunk
#   - chunk_overlap: overlap to preserve context

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunked_docs = text_splitter.split_documents(docs)

In [ ]:
# Embeddings: dense vectors for semantic similarity
# FAISS: vector DB for fast nearest-neighbor retrieval

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# HuggingFace embedding model
emb_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/multi-qa-mpnet-base-dot-v1",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},   # dot-product model, L2 index
)

# Build FAISS vector store from chunked_docs
vectordb = FAISS.from_documents(chunked_docs, emb_model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### The role of text splitters

Splitters keep chunks coherent while enforcing a size bound, carrying context across
boundaries so a chunk does not begin or end mid-thought.

In [ ]:
# Input:
#   - questions (list[str]): list of str
#   - vectordb: FAISS retriever
#   - llm: language model (Gemma-3, LangChain-wrapped)
#   - k: number of docs retrieved per query
#
# Process:
#   1. Retrieve top-k docs
#   2. Build context from retrieved docs
#   3. Generate answer with LLM
#   4. Post-process + collect doc IDs
# Output:
#   - rag_predicted_answers (list[str]): list of generated answers
#   - retrieved_ids (list[list[str]]): doc IDs retrieved

import re

def simple_rag(questions, vectordb, llm, k=5):

    rag_predicted_answers, retrieved_ids, retrieved_docs = [], [], []
    retriever = vectordb.as_retriever(search_kwargs={"k": k})

    for q in questions:
        # Use retriever to get relevant documents
        docs = retriever.invoke(q)

        context = "\n\n".join(d.page_content for d in docs)

        prompt = f"You are a helpful biomedical assistant. Answer the question using ONLY the context. If the context does not contain the answer, say you don't know. CONTEXT: {context} Question: {q} \n Answer:"

        answer = llm.invoke(prompt)

        answer_only = re.split(r"Answer:\s*", answer, maxsplit=1)[-1].strip()
        rag_predicted_answers.append(answer_only)

        ids = [d.metadata.get("doc_id") for d in docs]
        retrieved_ids.append(ids)
        retrieved_docs.append(docs[:k])

    return rag_predicted_answers, retrieved_ids, retrieved_docs

rag_predicted_answers, retrieved_ids, retrieved_docs = simple_rag(questions5, vectordb, llm, k=5)

/tmp/ipython-input-4189445179.py:28: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(q)


In [ ]:
#   - P (precision): how much of the predicted text is correct
#   - R (recall): how much of the reference was covered
#   - F1: harmonic mean of P and R

import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

simple_rag_scores = bertscore_report(rag_predicted_answers, answers5, "Simple RAG")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore - P: -0.010, R: -0.030, F1: -0.024


### Baseline retrieval scores

The scores are poor. The negative values indicate essentially no semantic overlap with the
gold answers.

In [ ]:
# Baseline display WITH sentence-level attribution
from IPython.display import display, HTML
import html, re
import math

def _sent_split(s: str):
    s = str(s).strip()
    parts = re.split(r'(?<=[.!?])\s+', s)
    return [p.strip() for p in parts if p.strip()]

def _norm_tokens(s: str):
    return [t for t in re.findall(r"[a-z0-9]+", s.lower()) if t]

def _overlap_score(a: str, b: str):
    # Jaccard on unigrams (simple, fast, robust)
    A, B = set(_norm_tokens(a)), set(_norm_tokens(b))
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

def _short(s, n):
    s = str(s).strip().replace("\n"," ")
    return s[:n-1]+"…" if len(s)>n else s

def render_rag_results_with_attribution(
    questions, preds, golds,
    retrieved_docs=None, retrieved_ids=None,
    max_preview_chars=260, per_sent_top=1
):
    cards=[]
    for i, q in enumerate(questions):
        pred = str(getattr(preds[i], "content", preds[i])) if i < len(preds) else ""
        gold = str(golds[i]) if i < len(golds) else ""

        # Build an array of (doc_id, sentences[]) for attribution if docs available
        doc_sents = []
        if retrieved_docs and i < len(retrieved_docs) and retrieved_docs[i]:
            for d in retrieved_docs[i]:
                if hasattr(d, "page_content"):
                    did = (getattr(d, "metadata", {}) or {}).get("doc_id", "NA")
                    sents = _sent_split(getattr(d, "page_content", ""))
                    if sents:
                        doc_sents.append((did, sents))

        # Per prediction sentence, pick highest-overlap supporting sentence across docs
        attrib_blocks = []
        pred_sents = _sent_split(pred)
        for ps in pred_sents:
            best = []
            for did, sents in doc_sents:
                # find best sentence in this doc
                local_best = max(sents, key=lambda s: _overlap_score(ps, s)) if sents else ""
                score = _overlap_score(ps, local_best) if local_best else 0.0
                best.append((score, did, local_best))
            if best:
                best.sort(reverse=True, key=lambda x: x[0])
                chosen = best[:per_sent_top]
                # render chosen supports (color intensity by score)
                sup_html = []
                for sc, did, sup in chosen:
                    shade = int(255 - min(1.0, sc) * 120)  # lower = darker for higher score
                    sup_html.append(
                        f"<div style='margin:6px 0;padding:8px;border-radius:8px;"
                        f"background: rgb({shade},{shade},{shade}); color:#111;'>"
                        f"<div style='font-size:12px;color:#222'><b>doc_id:</b> {html.escape(str(did))} "
                        f"<span style='opacity:.8'>(score {sc:.2f})</span></div>"
                        f"<div style='font-size:14px;line-height:1.35'>{html.escape(_short(sup, max_preview_chars))}</div>"
                        f"</div>"
                    )
                attrib_blocks.append(
                    f"<div style='margin-top:8px'>"
                    f"<div style='font-weight:700'>Pred sentence:</div>"
                    f"<div style='margin:4px 0 6px'>{html.escape(ps)}</div>"
                    f"{''.join(sup_html)}"
                    f"</div>"
                )

        # main card
        block = [f"""
        <div style="border:1px solid #ddd;border-radius:12px;padding:14px;margin:10px 0;">
          <div style="font-size:14px;color:#666;">Example {i+1}</div>
          <div style="font-size:16px;font-weight:700;margin-top:4px;">Question</div>
          <div>{html.escape(q)}</div>

          <div style="display:flex;gap:16px;margin-top:10px;">
            <div style="flex:1;">
              <div style="font-weight:700;">Predicted (stitched from retrieved chunks)</div>
              <div>{html.escape(pred) if pred else "<i>(empty)</i>"}</div>
            </div>
            <div style="flex:1;">
              <div style="font-weight:700;">Gold</div>
              <div>{html.escape(gold)}</div>
            </div>
          </div>
        """]
        # Attribution section
        if attrib_blocks:
            block.append("<div style='margin-top:12px;font-weight:700;'>Where the prediction likely came from</div>")
            block.append("".join(attrib_blocks))
        else:
            # fallback: list ids or short previews
            block.append("<div style='margin-top:12px;font-weight:700;'>Retrieved Evidence</div>")
            if retrieved_docs and i < len(retrieved_docs) and retrieved_docs[i]:
                for d in retrieved_docs[i]:
                    if hasattr(d, "page_content"):
                        did = (getattr(d, "metadata", {}) or {}).get("doc_id", "NA")
                        prev = _short(getattr(d, "page_content", ""), max_preview_chars)
                        block.append(
                            f"<div style='margin-top:8px;padding:10px;background:#fafafa;border:1px dashed #ddd;border-radius:8px;'>"
                            f"<div style='font-size:13px;color:#555;'><b>doc_id:</b> {html.escape(str(did))}</div>"
                            f"<div style='margin-top:4px;font-size:14px;line-height:1.4;'>{html.escape(prev)}</div>"
                            f"</div>"
                        )
            elif retrieved_ids and i < len(retrieved_ids):
                block.append(f"<div style='color:#555;'>doc_ids: {html.escape(str(retrieved_ids[i]))}</div>")
            else:
                block.append("<div style='color:#999;'>No retrievals recorded.</div>")

        block.append("</div>")
        cards.append("".join(block))
    display(HTML("".join(cards)))

In [ ]:
render_rag_results_with_attribution(
    questions5, rag_predicted_answers, answers5,
    retrieved_docs=retrieved_docs,  # list[list[Document]]
    per_sent_top=1                  # show top-1 supporting sentence per pred sentence
)

## Hybrid retrieval, reranking and compression

The full pipeline: dense and sparse retrieval combined, a cross-encoder reranking the
candidates, contextual compression reducing each snippet to its relevant sentences, and an
adaptive retry that widens the search when the model abstains.

In [ ]:
import re
import random
from typing import List, Any, Tuple, Optional
from dataclasses import dataclass

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Reuse the weights already resident on the GPU rather than loading a second copy.
model = llm.pipeline.model
tokenizer = llm.pipeline.tokenizer

gen = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,           # deterministic; flip to True if you want sampling
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=gen)  # <- now llm.invoke(prompt) works

# ---- Configuration knobs ----
CFG = {
    # Embedders & rerankers
    "dense_embedder": "pritamdeka/S-Biomed-Roberta-snli-multinli-stsb",
    "rerankers_try": [
        "ncbi/MedCPT-Cross-Encoder",
        "cross-encoder/ms-marco-MiniLM-L-12-v2",
        "BAAI/bge-reranker-base",
    ],
    # LLM & tokenizer
    "tokenizer": "google/gemma-3-1b-it",
    # Retrieval sizes
    "dense_k": 80,
    "bm25_k": 120,
    "rerank_k": 8,
    "rerank_k_retry": 12,
    # Compression thresholds
    "tau": 0.5,
    "tau_retry": 0.47,
    "max_sents": 8,
    "max_sents_retry": 10,
    # Context token cap
    "max_ctx_tokens": 1400,
    # Scoring
    "idk_str": "I don't know.",
    # Device flags (let backends pick GPU if available)
    "device": DEVICE,
    "seed": 42,
}

random.seed(CFG["seed"])

Device set to use cuda:0


### Data preparation

BioASQ ships passages and IDs as plain JSON. Wrapping them as LangChain `Document`
objects with `doc_id` in the metadata keeps retrievals traceable back to the source
passage.

In [ ]:
# - Convert raw JSON records into Document objects
# - Attach doc_id to metadata so we can track provenance later

def build_base_docs(bio_corpus) -> List[Document]:
    # bio_corpus['test'] = [{"passage": "...", "id": ...}, ...]
    return [
        Document(page_content=rec["passage"], metadata={"doc_id": rec["id"]})
        for rec in bio_corpus["test"]
    ]

base_docs = build_base_docs(bio_corpus)

Whole abstracts are too coarse a retrieval unit, so each one is split into snippets of
two to three sentences.

In [ ]:
# - Break each abstract into 2–3 sentence chunks
# - This improves retriever granularity

# Provided helper: simple regex-based sentence splitter
_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

def sent_tokenize_quick(text: str) -> List[str]:
    sents = [s.strip() for s in _SENT_SPLIT.split(text) if s.strip()]
    return sents if sents else [text.strip()]

def make_snippets(docs: List[Document], max_sents_per_snip: int = 3) -> List[Document]:
    out = []
    for d in docs:
        sents = sent_tokenize_quick(d.page_content)
        buf = []
        for s in sents:
            buf.append(s)
            if len(buf) >= max_sents_per_snip:
                out.append(Document(page_content=" ".join(buf), metadata=d.metadata))
                buf = []
        if buf:
            out.append(Document(page_content=" ".join(buf), metadata=d.metadata))
    return out

snippet_docs = make_snippets(base_docs, max_sents_per_snip=3)

### Chunk-size trade-offs

**Recall** — long documents give higher recall, but the relevant span can be buried.
Very short snippets lose recall when the key evidence spans several sentences and gets
split across chunks.

**Precision** — long documents carry many unrelated sentences and so score lower;
short snippets are focused and score higher per chunk.

**Answer quality** — long documents risk exhausting the context window and distracting
the model. Very short snippets fit cleanly but can drop qualifiers such as species or
dosage.

Two to three sentences preserves local context and keeps retrieval granular while holding
token usage down, which also reduces fragmentation.

### Retrieval setup

Two complementary retrievers: a dense semantic retriever over biomedical SBERT embeddings
indexed in FAISS, and a sparse lexical retriever using BM25.

In [ ]:
# - Dense: FAISS (Facebook AI Similarity Search) index with biomedical SBERT (sentence-BERT) embeddings
# - Sparse: BM25 (Best Matching 25) lexical retriever
# - Output: two retrievers for hybrid recall

def bm25_preprocess(text: str) -> List[str]:
    """Lowercase and drop punctuation, so `PCSK9,` and `pcsk9` match."""
    return re.findall(r"[a-z0-9]+", text.lower())


def build_retrievers(snippet_docs: List[Document]):
    emb = HuggingFaceEmbeddings(
        model_name=CFG["dense_embedder"],
        model_kwargs={"device": CFG["device"]},
        encode_kwargs={"normalize_embeddings": True}
    )
    # Vector store + dense retriever
    vectordb = FAISS.from_documents(snippet_docs, emb)
    dense_retriever = vectordb.as_retriever(search_kwargs={"k": CFG["dense_k"]})

    # Sparse retriever (pure BM25 over snippet text)
    bm25 = BM25Retriever.from_documents(
        snippet_docs, k=CFG["bm25_k"], preprocess_func=bm25_preprocess
    )
    return dense_retriever, bm25

dense_retriever, bm25 = build_retrievers(snippet_docs)

In [ ]:
# - Pull candidates from both retrievers
# - Slice BM25 by bm25_fetch_k (lexical recall budget)
# - Deduplicate by (doc_id, first_120_chars) to avoid near-duplicates
# - Output: merged candidate list for reranking

def hybrid_candidates(q: str, dense_retriever, bm25, bm25_fetch_k: int) -> List[Document]:
    dense = dense_retriever.invoke(q) or []
    bm25.k = bm25_fetch_k          # BM25Retriever defaults to k=4
    sparse = bm25.invoke(q)[:bm25_fetch_k] or []
    seen, merged = set(), []
    for d in (dense + sparse):
        key = (d.metadata.get("doc_id", "NA"), d.page_content[:120])
        if key not in seen and d.page_content.strip():
            seen.add(key); merged.append(d)
    return merged

### Dense retrieval against BM25 on biomedical text

**Dense** captures semantic similarity and synonymy, but depends heavily on its training
domain and degrades off-topic.

**BM25** handles exact terminology well, including gene and drug names and abbreviations,
but struggles with synonyms, paraphrase and morphological variation.

BioASQ needs both: it mixes controlled vocabulary such as chemical names with paraphrased
natural language, and many answers turn on exact tokens as well as semantic relations.
Running both raises recall before the later filtering stages.

### Reranking and compression

Hybrid retrieval is deliberately broad, so its output is reranked by a cross-encoder and
then compressed down to the sentences that actually bear on the question.

In [ ]:
# - Load a pretrained reranker (biomedical if available)
# - Score each (query, snippet) pair with the reranker
# - Sort candidates by score, keep top-k for downstream QA

def load_reranker() -> CrossEncoder:
    last_err = None
    for name in CFG["rerankers_try"]:
        try:
            ce = CrossEncoder(name, device=CFG["device"], max_length=512)
            print("Loaded reranker:", name)
            return ce
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Failed to load all rerankers. Last error: {last_err}")

def rerank_snippets(q: str, docs: List[Document], reranker: CrossEncoder, k_final: int) -> List[Document]:
    if not docs:
        return []

    pairs = [(q, d.page_content) for d in docs]

    scores = reranker.predict(pairs, batch_size=32, convert_to_numpy=True)

    order = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)][:k_final]

    return [docs[i] for i in order]

reranker = load_reranker()

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loaded reranker: ncbi/MedCPT-Cross-Encoder


### What reranking fixes

Hybrid recall is broad by design, so it returns redundant and irrelevant snippets.
Reranking corrects term-only BM25 matches that are out of context, semantic drift from the
dense retriever, and near-duplicates, by pushing better-supported snippets to the top.

In [ ]:
# - Break candidate snippets into sentences
# - Encode query + sentences with a lightweight biomed encoder
# - Score by cosine similarity, keep top sentences above tau
# - Return compressed context (or insufficient)

# Lightweight encoder for compression
_comp_enc = SentenceTransformer(CFG["dense_embedder"], device=CFG["device"])

def compress(q: str, docs_kept: List[Document], max_sents: int, tau: float):
    candidates, cand_ids = [], []
    for d in docs_kept:
        sents = sent_tokenize_quick(d.page_content)
        for s in sents:
            if s:
                candidates.append(s)
                cand_ids.append(d.metadata.get("doc_id", "NA"))
    if not candidates:
        return [], [], True    # no sentences → immediate insufficient

    q_emb = _comp_enc.encode(q, normalize_embeddings=True)
    s_emb = _comp_enc.encode(candidates, normalize_embeddings=True)

    # Compute similarity scores (cosine = dot product since normalized)
    sims = (q_emb @ s_emb.T).ravel()

    idx = [i for i, _ in sorted(enumerate(sims), key=lambda x: x[1], reverse=True)]

    picked, seen = [], set()
    for i in idx:
        if sims[i] < tau:
            break

        key = candidates[i].strip().lower()
        if key in seen:
            continue
        seen.add(key)
        picked.append(i)

        if len(picked) >= max_sents:
            break

    # Regroup by source document in reading order. Similarity order interleaves
    # sentences from unrelated abstracts, which reads as a bag of fragments.
    doc_rank = {}
    for i in picked:
        doc_rank.setdefault(cand_ids[i], len(doc_rank))
    picked.sort(key=lambda i: (doc_rank[cand_ids[i]], i))

    picked_sents = [candidates[i] for i in picked]
    picked_ids = [cand_ids[i] for i in picked]

    return picked_sents, picked_ids, not picked_sents

### Threshold and sentence-count trade-offs

**τ too high** filters away supporting sentences, leaving too little evidence and
producing more abstentions. **τ too low** keeps noisy sentences, diluting the context and
raising the chance of hallucination.

**Larger `max_sents`** gives more coverage at the cost of distractors and token bloat.
**Smaller `max_sents`** keeps the context tight but can drop qualifiers the answer
depends on.

### Answer generation

Generation is constrained to the compressed context, with an explicit abstention option
when the evidence is too weak to support an answer.

The prompt constrains the model to the retrieved context and permits "I don't know".
Context is token-truncated to a fixed budget, prompt fragments echoed into the output are
stripped, and refusals are detected so the retry policy can act on them.

In [ ]:
# - Truncate context to a fixed token budget
# - Force “use ONLY the context” + abstention policy
# - Clean echoed prompt text from the model output

tokenizer = AutoTokenizer.from_pretrained(CFG["tokenizer"])

def token_truncate(text: str, tokenizer, max_tokens: int) -> str:
    ids = tokenizer.encode(text, add_special_tokens=False)
    return tokenizer.decode(ids[:max_tokens])

def clean_answer_echo(ans: str, q: str) -> str:
    t = str(ans).strip()
    if "Answer:" in t:
        t = t.split("Answer:", 1)[-1].strip()
    t = re.sub(r"^Answer ONLY.*?Answer:\s*", "", t, flags=re.IGNORECASE|re.DOTALL)
    t = re.sub(r"Question:.*?Context:.*?Answer:\s*", "", t, flags=re.IGNORECASE|re.DOTALL)
    t = re.sub(rf"^{re.escape(q)}[\s\?]*", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"\s+", " ", t).strip()
    return t

_REFUSAL_MARKERS = (
    "i don't know", "i do not know", "i dont know",
    "insufficient evidence", "not enough evidence", "no evidence",
    "cannot answer", "can't answer", "unable to answer",
    "does not contain", "doesn't contain", "not contain the answer",
    "not provided in the context", "not mentioned in the context",
)


def is_refusal(txt: str) -> bool:
    t = str(txt).lower().strip()
    if not t:
        return True
    return any(m in t for m in _REFUSAL_MARKERS)

def answer_with_context(q: str, sents: List[str], ids_used: List[Any], max_ctx_tokens: int):
    ctx = "\n".join(s.strip() for s in sents) if sents else ""
    if ctx:
        ctx = token_truncate(ctx, tokenizer, max_ctx_tokens)
    prompt = (
        "Use ONLY the context to answer concisely. "
        "If the answer is unclear or missing, say \"I don't know.\""
        f"\n\nQuestion: {q}\n\nContext:\n{ctx or '(EMPTY)'}\n\nAnswer:"
    )
    out = llm.invoke(prompt)
    text = getattr(out, "content", out)
    pred = clean_answer_echo(text, q)
    return pred, ids_used

### Why abstention matters

Abstention is a safety guardrail for biomedical answers, trading recall for precision:
"I don't know" is preferable to a confident wrong answer.

Letting the model draw freely on outside knowledge invites hallucinated facts and
fabricated citations, over-generalisation, and loss of traceability back to the evidence.

In [ ]:
# - Run hybrid retrieve → rerank → compress → answer
# - If refusal ("I don't know"), retry ONCE with a relaxed policy

def qa_pipeline(questions: List[str],
                dense_retriever,
                bm25,
                reranker: CrossEncoder,
                k_final: int,
                k_retry: int,
                tau: float,
                tau_retry: float,
                max_sents: int,
                max_sents_retry: int,
                max_ctx_tokens: int):
    preds, ids = [], []
    for q in questions:
        # S3.2 Hybrid Retrieve
        cands = hybrid_candidates(q, dense_retriever, bm25, CFG["bm25_k"])

        # S4: Rerank
        kept  = rerank_snippets(q, cands, reranker, k_final)

        # S5: Compress
        sents, ids_used, insufficient = compress(q, kept, max_sents=max_sents, tau=tau)

        if insufficient:
            pred, used = CFG["idk_str"], []
        else:
            pred, used = answer_with_context(q, sents, ids_used, max_ctx_tokens=max_ctx_tokens)

        # Thin evidence and an outright refusal call for the same response: widen the net.
        if insufficient or is_refusal(pred):

            kept_retry = rerank_snippets(q, cands, reranker, k_retry)

            sents_retry, ids_retry, insufficient2 = compress(
                q, kept_retry, max_sents=max_sents_retry, tau=tau_retry
            )
            if not insufficient2:
                pred2, used2 = answer_with_context(q, sents_retry, ids_retry, max_ctx_tokens=max_ctx_tokens)

                if not is_refusal(pred2):
                    pred, used = pred2, used2

        if is_refusal(pred):
            pred = CFG["idk_str"]

        preds.append(pred); ids.append(used)
    return preds, ids

# Hybrid QA pipeline using helpers
preds_pipeline, ids_pipeline = qa_pipeline(
    questions=questions5,
    dense_retriever=dense_retriever,
    bm25=bm25,
    reranker=reranker,
    k_final=CFG["rerank_k"],         # rerank_snippets
    k_retry=CFG["rerank_k_retry"],   # adaptive retry
    tau=CFG["tau"],                  # compress threshold
    tau_retry=CFG["tau_retry"],      # retry threshold
    max_sents=CFG["max_sents"],      # compression sentence cap
    max_sents_retry=CFG["max_sents_retry"],
    max_ctx_tokens=CFG["max_ctx_tokens"],  # answer_with_context token budget
)

In [ ]:
# - Visualize baseline vs reranker outputs side by side
# - Include question, gold answer, predictions, and supporting doc IDs
# - This helps us compare retrieval+QA variants directly

from IPython.display import display, HTML
import html

def render_baseline_vs_variant(questions, base_preds, variant_preds, golds,
                               base_ids, variant_ids):
    cards = []
    for i, (q, bpred, vpred, g, bids, vids) in enumerate(
        zip(questions, base_preds, variant_preds, golds, base_ids, variant_ids)
    ):
        cards.append(f"""
        <div style="border:1px solid #ddd;border-radius:10px;padding:12px;margin:10px 0;">
          <div style="color:#666">Example {i+1}</div>

          <div style="font-weight:700;margin-top:4px;">Question</div>
          <div>{html.escape(q)}</div>

          <div style="display:flex;gap:16px;margin-top:10px;">
            <div style="flex:1;">
              <div style="font-weight:700;">Baseline Prediction</div>
              <div>{html.escape(str(bpred)) if bpred else "<i>(empty)</i>"}</div>
            </div>
            <div style="flex:1;">
              <div style="font-weight:700;">Pipeline Prediction</div>
              <div>{html.escape(str(vpred)) if vpred else "<i>(empty)</i>"}</div>
            </div>
            <div style="flex:1;">
              <div style="font-weight:700;">Gold</div>
              <div>{html.escape(str(g))}</div>
            </div>
          </div>

          <div style="font-weight:700;margin-top:10px;">Doc IDs</div>
          <div style="font-size:13px;"><b>Baseline:</b> {html.escape(str(bids))}</div>
          <div style="font-size:13px;"><b>Pipeline:</b> {html.escape(str(vids))}</div>
        </div>
        """)
    display(HTML("".join(cards)))

# Render comparison
render_baseline_vs_variant(
    questions5,
    rag_predicted_answers,   # baseline predictions
    preds_pipeline,          # full pipeline (hybrid + rerank + compress + retry + answer)
    answers5,                # gold labels
    retrieved_ids,           # baseline doc IDs
    ids_pipeline             # pipeline doc IDs
)

### Reranked pipeline against the baseline

The reranking pipeline reduced wrong answers and improved correctness overall. One
example turned a hallucination into a correct answer. Abstentions were better calibrated
where evidence was thin, giving higher precision than the baseline.

### LLM-as-a-judge evaluation

Baseline and full-pipeline outputs were scored by a stronger external model, weighted so
that abstention is penalised less than a confident wrong answer.

### Judge results

[Full judge transcript](https://chatgpt.com/share/68d61b32-247c-800d-9b89-26e6a57e34bd)

The advanced pipeline is mixed. It answers binary and mechanism questions correctly
(examples 1, 2 and 4), but drifts on formatting and intent in example 3, and misses a rare
term in example 5.